In [5]:
include("../src/main.jl");

In [6]:
dump_filepath = "../src/models/iharm3dDumps/dump_001.h5";
#dump_filepath = "../../../../Downloads/torus.out0.00356.h5";

In [7]:
#TODO: put this in reading file
const N1 = 128
const N2 = 64
const N3 = 32

const METRIC = "FMKS" #FMKS or MKS TODO: prob have to be read from file
const trat_large = 20. #TODO: prob have to be read from file
const trat_small = 1. #TODO: prob have to be read from file
const beta_crit = 1.0 #TODO: prob have to be read from file
const game = (4. /3.)  # Ion adiabatic index  TODO: prob have to be read from file
const gamp = (5. /3.)  # Electron adiabatic index TODO: prob have to be read from file
const gam = (13. /9.)  # Total adiabatic index TODO: prob have to be read from file
const Ne_factor = 1.0  # Scaling factor for electron number density TODO: prob have to be read from file
const rmin_geo = 1.00187575798832   #TODO: Has to be read from file as Rin and compared to the value chosen
const rmax_geo = 100. #TODO: Has to be read from file as Rin and compared to the value chosen
const th_beg = 1.74e-2 #TODO: Idk where this comes from, check ipole source code
const sigma_cut = 1.0 #TODO: maybe put it somewhere else?
const sigma_cut_high = -1.0
const startx::MVec4 = [0, 1.874000951149813e-03, 0, 0]#TODO: prob have to be read from file
const stopx::MVec4 = [1, 6.907755278982138e+00, 1, 2 * π]#TODO: prob have to be read from file
const dx::MVec4 =[0, 5.395219748461709e-02, 1.562500000000000e-02, 1.963495408493621e-01]
const bhspin = 0.9375 #TODO: prob have to be read from file
const hslope = 0.3 #TODO: prob have to be read from file


0.3

In [8]:
simulation_data = load_data(dump_filepath);

Loading data from '../src/models/iharm3dDumps/dump_001.h5' into 'iharm' module...
Primitives successfully loaded. Dimensions: (128, 64, 32)
Calculating physical quantities...
Using mixed tp_over_te with trat_small = 1.0, trat_large = 20.0, and beta_crit = 1.0
All primitives successfully loaded. Dimensions: (128, 64, 32)


In [9]:
#Analytic parameters

#Setting up the parameters
#Observer distance in Rg
const ro = 1000.0
#Observer inclination in degrees
const th = 80.0

#Observer azimuth in degrees
const phi = 0.0

# Number of pixels in the x and y direction. The number of geodesics calculated will be res^2
const res = 80
const pixels_x = 80
const pixels_y = 80
# Distance to the source in parsecs
const SourceD = 16.9e6 * PC
const Rout = 1000.0
const Rstop = 100.0
const Rh = 1 + sqrt(1. - bhspin * bhspin);

#Check if these are correct
#const cstartx = [0.0, log(Rh), 0.0, 0.0]#TODO: prob have to be read from file
const cstartx = MVec4(0.0, log(Rh), 0.0, 0.0)#TODO: prob have to be read from file
const cstopx = MVec4(0.0, log(Rout), 1.0, 2.0 * π)#TODO: prob have to be read from file

# Frequency observed by the camera in Hz
const freq = 230e9;

# Size of the screen in Rg in both directions
const DXsize = SourceD/L_unit/MUAS_PER_RAD * 160
const DYsize = SourceD/L_unit/MUAS_PER_RAD * 160
# Observer fov in radians (this can be understood as size of the plane camera sees over the distance ro)
# This should be atan, but for small angles it is approximately equal to the angle itself
const fovx = DXsize/ro
const fovy = DYsize/ro
const xoff = 0.0
const yoff = 0.0


0.0

In [10]:
include("../src/main.jl");
# Find camera in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, bhspin, Rout))

# Scales the intensity of each pixel by the real size of each pixel
scale_factor = CalculateScaleFactor(DXsize, DYsize, pixels_x, pixels_y, SourceD, L_unit)
println("scale_factor = $scale_factor")
const maxnstep = 15000
# Generate geodesics
println("Utilizing $(Threads.nthreads()) threads for geodesic calculation.")
freq_unitless = freq * HPL/(ME * CL * CL)  # Convert frequency to unitless
Image = zeros(Float64, pixels_x, pixels_y)
for i in 0:(pixels_x - 1)
    println("Processing row $i out of $(pixels_x)")

    Threads.@threads for j in 0:(pixels_y - 1)
        traj = Vector{OfTraj}()
        sizehint!(traj, maxnstep)
        nstep = get_pixel(traj, i, j, Xcamera, maxnstep, fovx, fovy, freq_unitless, pixels_x, pixels_y, bhspin, Rh, Rout, Rstop, xoff, yoff)

        resize!(traj, length(traj))
        integrate_emission!(traj, length(traj), Image, i + 1, j + 1, freq, bhspin, simulation_data)
    end
end
Image *= freq^3;

scale_factor = 2.350438638182906
Utilizing 8 threads for geodesic calculation.
Processing row 0 out of 160
Processing row 1 out of 160
Processing row 2 out of 160
Processing row 3 out of 160
Processing row 4 out of 160
Processing row 5 out of 160
Processing row 6 out of 160
Processing row 7 out of 160
Processing row 8 out of 160
Processing row 9 out of 160
Processing row 10 out of 160
Processing row 11 out of 160
Processing row 12 out of 160
Processing row 13 out of 160
Processing row 14 out of 160
Processing row 15 out of 160
Processing row 16 out of 160
Processing row 17 out of 160
Processing row 18 out of 160
Processing row 19 out of 160
Processing row 20 out of 160
Processing row 21 out of 160
Processing row 22 out of 160
Processing row 23 out of 160
Processing row 24 out of 160
Processing row 25 out of 160
Processing row 26 out of 160
Processing row 27 out of 160
Processing row 28 out of 160
Processing row 29 out of 160
Processing row 30 out of 160
Processing row 31 out of 160
Pro

In [7]:
OutputStokesParameters(Image, freq, scale_factor, res, SourceD)

Image processing complete. Calculating total flux and averages...
Scale = 2.350438638182906e+00
imax = 27, jmax = 89, Imax = 0.006416052023077334, Iavg = 0.0014917239582839844
Using freq_cgs = 2.3e11, Ftot = 89.75886410377706
nuLnu = 7.054880559756697e42


In [11]:
using DelimitedFiles

writedlm("./Image.txt", Image)
